# Geographic distribution — SIP by state + T30 vs B30 city tier
Horizontal bar chart: total **SIP** amount by **state**.
Pie chart: split of SIP amount between **T30** and **B30** city tiers.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


# --- Repo-root detection (so notebook works from any notebook working directory) ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()


def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'investor_transactions_clean.csv').exists():
            return cand

        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'investor_transactions_clean.csv').exists():
                return parent

        cand = cand.parent

    return start.parent


_REPO_ROOT = _find_repo_root(_HERE)
DATA_PATH = _REPO_ROOT / 'Data' / 'processed' / 'investor_transactions_clean.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing file: {DATA_PATH.resolve()}')


df = pd.read_csv(DATA_PATH)

# Basic cleanup / typing
df['transaction_type'] = df['transaction_type'].astype(str)
df['state'] = df['state'].astype(str)
df['city_tier'] = df['city_tier'].astype(str)
df['amount_inr'] = pd.to_numeric(df['amount_inr'], errors='coerce')
df = df.dropna(subset=['amount_inr', 'state', 'city_tier', 'transaction_type']).copy()

df_sip = df.loc[df['transaction_type'] == 'SIP'].copy()

# Convert to Cr (₹ crore)
df_sip['amount_cr'] = df_sip['amount_inr'] / 1e7

print('Total SIP rows:', len(df_sip))
print('States:', df_sip['state'].nunique())
print('City tiers:', sorted(df_sip['city_tier'].unique()))


Total SIP rows: 19716
States: 12
City tiers: ['B30', 'T30']


In [2]:
# --- 1) Horizontal bar chart: SIP amount by state ---
state_agg = (
    df_sip.groupby('state', as_index=False)['amount_cr']
    .sum()
    .sort_values('amount_cr', ascending=False)
)

# Keep chart readable: top 15 states + 'Others' (if needed)
TOP_N = 15
if len(state_agg) > TOP_N:
    top = state_agg.head(TOP_N).copy()
    others_sum = state_agg.iloc[TOP_N:]['amount_cr'].sum()
    top = pd.concat([top, pd.DataFrame([{'state': 'Others', 'amount_cr': others_sum}])], ignore_index=True)
    state_agg = top

fig_bar = go.Figure()

fig_bar.add_trace(
    go.Bar(
        x=state_agg['amount_cr'],
        y=state_agg['state'],
        orientation='h',
        marker=dict(color='#4c78a8'),
        hovertemplate='State=%{y}<br>SIP=%{x:,.2f} Cr<extra></extra>',
    )
)

fig_bar.update_layout(
    title='SIP amount by state (horizontal bar) — total SIPs',
    template='plotly_white',
    xaxis_title='SIP amount (₹ crore)',
    yaxis_title='State',
    height=650,
    margin=dict(l=120, r=30, t=60, b=40),
)

fig_bar.show()


In [3]:
# --- 2) Pie chart: T30 vs B30 city tier split ---
tier_order = ['T30', 'B30']
tier_df = df_sip.loc[df_sip['city_tier'].isin(tier_order)].copy()
tier_agg = (
    tier_df.groupby('city_tier', as_index=False)['amount_cr']
    .sum()
    .set_index('city_tier')
    .reindex(tier_order)
    .reset_index()
)
tier_agg['amount_cr'] = tier_agg['amount_cr'].fillna(0.0)

fig_pie = go.Figure()

fig_pie.add_trace(
    go.Pie(
        labels=tier_agg['city_tier'],
        values=tier_agg['amount_cr'],
        hole=0.35,
        sort=False,
        textinfo='percent+label',
        hovertemplate='City tier=%{label}<br>SIP=%{value:,.2f} Cr<extra></extra>',
        marker=dict(colors=['#d62728', '#4c78a8']),
    )
)

total_cr = float(tier_agg['amount_cr'].sum())
fig_pie.update_layout(
    title=f'SIP amount split by city tier (T30 vs B30) — total: ₹{total_cr:,.2f} Cr',
    template='plotly_white',
    height=520,
    margin=dict(l=30, r=30, t=60, b=30),
)

fig_pie.show()
